# 怎样从一批笔记中找到相关段落？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

[{'doc_id': 'D1', 'start': 0, 'end': 40, 'text': '实验室预约需要提前一天提交申请。每次预约最长两小时。取消预约请在开始前四小时通知', 'score': 0.37936034177912303}, {'doc_id': 'D6', 'start': 30, 'end': 53, 'text': '本。GPU预约冲突时优先安排已批准的课程实验。', 'score': 0.048452088425244025}, {'doc_id': 'D6', 'start': 0, 'end': 40, 'text': '服务器训练任务最长运行六小时。提交前需要记录随机种子和依赖版本。GPU预约冲突时', 'score': 0.03538279605858653}]
{
  "recall_at_1": 1.0,
  "recall_at_3": 1.0,
  "mrr_at_3": 1.0,
  "chunks": 12,
  "evaluation_questions": 6,
  "elapsed_seconds": 0.783
}


## 1. 补全：重叠切分
窗口 6、重叠 2，核对 ABCDEFGHIJ 的片段以及非法参数。

In [2]:
parts = chunk_text('ABCDEFGHIJ', 6, 2)
assert [p[2] for p in parts] == ['ABCDEF', 'EFGHIJ']
try:
    chunk_text('abc', 2, 2)
except ValueError:
    pass
else:
    raise AssertionError('非法参数未拒绝')
print(parts)

[(0, 6, 'ABCDEF'), (4, 10, 'EFGHIJ')]


## 2. 修改：无匹配处理
检查空查询与完全不在语料里的查询。

In [3]:
assert retrieve('') == []
assert retrieve('zzzzzz') == []
assert retrieve('实验室预约', 1)[0]['doc_id'] == 'D1'
print('通过')

通过


## 3. 迁移：加入新文档
新增会议室规则并重建独立索引，核对新来源可找到。

In [4]:
new_docs = {**documents, 'D7': '会议室预约必须填写参会人数。会议结束后应关闭投影仪。'}
v = TfidfVectorizer(analyzer='char', ngram_range=(2, 4))
ids = list(new_docs)
m = v.fit_transform(list(new_docs.values()))
score = (m @ v.transform(['会议室参会人数']).T).toarray().ravel()
assert ids[int(score.argmax())] == 'D7'
print(ids[int(score.argmax())])

D7
